In [ ]:
from google.colab import drive
import os

# 1. Montar o Google Drive
drive.mount('/content/drive')

In [ ]:
print(os.listdir('/content/drive/MyDrive/Imagens Barreiras'))

In [ ]:
import os

# Definindo o caminho da pasta que estava faltando
PASTA = '/content/drive/MyDrive/Imagens Barreiras'

print("Conteúdo da pasta principal:")
print(os.listdir(PASTA))

# Ajustando os nomes das subpastas conforme listado no seu Drive
pasta_com_lona = os.path.join(PASTA, 'Barreiras com lona')
pasta_sem_lona = os.path.join(PASTA, 'Barrairas sem lona')

print("Quantidade de arquivos:")
print(f"Com lona: {len(os.listdir(pasta_com_lona))}")
print(f"Sem lona: {len(os.listdir(pasta_sem_lona))}")

In [ ]:
!pip -q install ultralytics scikit-learn

In [ ]:
PASTA = '/content/drive/MyDrive/NomeDoAtalho/barreiras'   # seu caminho
CLASSES = ['com_lona', 'sem_lona']
print(PASTA, CLASSES)

In [ ]:
import os, hashlib, collections
from PIL import Image

# Ajustando para os caminhos reais do seu Drive
PASTA = '/content/drive/MyDrive/Imagens Barreiras'
CLASSES = ['Barreiras com lona', 'Barrairas sem lona']

EXT = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
fotos = []
for c in CLASSES:
    pasta_c = os.path.join(PASTA, c)
    assert os.path.isdir(pasta_c), f'Pasta não encontrada: {pasta_c}'
    for nome in sorted(os.listdir(pasta_c)):
        if os.path.splitext(nome)[1].lower() in EXT:
            fotos.append((os.path.join(pasta_c, nome), c))

validas, ruins, vistos = [], [], {}
for caminho, c in fotos:
    try:
        with Image.open(caminho) as im:
            im.verify()
    except Exception:
        ruins.append(caminho); continue
    h = hashlib.md5(open(caminho, 'rb').read()).hexdigest()
    if h in vistos:
        print('Repetida:', os.path.basename(caminho), '=', os.path.basename(vistos[h][0]), f'({vistos[h][1]} / {c})')
        continue
    vistos[h] = (caminho, c)
    validas.append((caminho, c))

print('\nFotos por classe:', dict(collections.Counter(c for _, c in validas)))
if ruins: print('Arquivos que não abriram (ignorados):', ruins)

In [ ]:
import shutil, random, numpy as np
from sklearn.model_selection import StratifiedGroupKFold
from ultralytics import YOLO

SEMENTE = 42
K = 5
EPOCAS = 30
TAMANHO = 224

caminhos = [p for p, _ in validas]
rotulos  = np.array([0 if c == CLASSES[0] else 1 for _, c in validas])      # 0 = com lona, 1 = sem lona
grupos   = [os.path.basename(p).split('_')[0] if '_' in os.path.basename(p) else os.path.basename(p) for p in caminhos]
IDX_SEM  = sorted(CLASSES).index(CLASSES[1])     # posição da classe "sem lona" na saída do YOLO (ordem alfabética)

def montar_dobra(nome, idx_treino, idx_val):
    raiz = f'/content/cv/{nome}'
    shutil.rmtree(raiz, ignore_errors=True)
    por_classe = {c: [i for i in idx_treino if rotulos[i] == c] for c in (0, 1)}
    maior = max(len(v) for v in por_classe.values())
    rng = random.Random(SEMENTE)
    for c, lista in por_classe.items():
        destino = os.path.join(raiz, 'train', CLASSES[c])
        os.makedirs(destino, exist_ok=True)
        if len(lista) > 0:
            escolhidos = list(lista) + [rng.choice(lista) for _ in range(maior - len(lista))]
        else:
            escolhidos = []  # Evita erro caso a dobra não tenha amostras desta classe no treino
        for n, i in enumerate(escolhidos):
            ext = os.path.splitext(caminhos[i])[1]
            shutil.copy(caminhos[i], os.path.join(destino, f'{n:04d}{ext}'))
    for i in idx_val:
        destino = os.path.join(raiz, 'val', CLASSES[rotulos[i]])
        os.makedirs(destino, exist_ok=True)
        shutil.copy(caminhos[i], os.path.join(destino, os.path.basename(caminhos[i])))
    return raiz

def treinar(raiz, nome, epocas=EPOCAS):
    modelo = YOLO('yolov8n-cls.pt')
    modelo.train(data=raiz, epochs=epocas, imgsz=TAMANHO, batch=16, seed=SEMENTE, patience=10,
                 project='/content/runs', name=nome, exist_ok=True, plots=False, verbose=False,
                 fliplr=0.5, degrees=10, hsv_v=0.4, hsv_s=0.4, scale=0.3, erasing=0.2)
    return YOLO(f'/content/runs/{nome}/weights/best.pt')

def prever_lote(modelo, lista):
    saida = []
    for r in modelo.predict(lista, imgsz=TAMANHO, verbose=False):
        saida.append(r.probs.data.cpu().numpy()[IDX_SEM])     # probabilidade de "sem lona"
    return np.array(saida)

prob_sem = np.zeros(len(caminhos))
skf = StratifiedGroupKFold(n_splits=K, shuffle=True, random_state=SEMENTE)
for k, (tr, va) in enumerate(skf.split(caminhos, rotulos, grupos), start=1):
    raiz = montar_dobra(f'dobra{k}', tr, va)
    modelo = treinar(raiz, f'dobra{k}')
    prob_sem[va] = prever_lote(modelo, [caminhos[i] for i in va])
    acertos = ((prob_sem[va] >= 0.5).astype(int) == rotulos[va]).mean()
    print(f'Parte {k}/{K}: {len(va)} fotos de teste, acerto {acertos:.0%}')

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt

LIMIAR = 0.5
previsto = (prob_sem >= LIMIAR).astype(int)
print(classification_report(rotulos, previsto, target_names=['com_lona', 'sem_lona'], digits=3))
print('AUC (separação entre as classes, 1,0 = perfeita):', round(roc_auc_score(rotulos, prob_sem), 3))

cm = confusion_matrix(rotulos, previsto)
fig, ax = plt.subplots(figsize=(4.2, 3.6))
ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_xticklabels(['com_lona', 'sem_lona']); ax.set_yticks([0, 1]); ax.set_yticklabels(['com_lona', 'sem_lona'])
ax.set_xlabel('Previsto'); ax.set_ylabel('Real'); ax.set_title('Matriz de confusão (validação cruzada)')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=14)
plt.tight_layout(); plt.show()

erradas = [(os.path.basename(caminhos[i]), CLASSES[rotulos[i]], round(float(prob_sem[i]), 2)) for i in range(len(caminhos)) if previsto[i] != rotulos[i]]
print('Fotos que o modelo errou (arquivo, classe real, prob. de sem_lona):')
for e in erradas: print('  ', e)

In [ ]:
from sklearn.model_selection import train_test_split
idx = np.arange(len(caminhos))
tr, va = train_test_split(idx, test_size=0.15, stratify=rotulos, random_state=SEMENTE)
raiz = montar_dobra('final', tr, va)
final = treinar(raiz, 'final', epocas=EPOCAS)

destino = '/content/drive/MyDrive/classificador_lona.pt'
shutil.copy('/content/runs/final/weights/best.pt', destino)
print('Modelo salvo em:', destino)

In [ ]:
def prever_lona(caminho_foto, modelo=final, limiar=LIMIAR):
    r = modelo.predict(caminho_foto, imgsz=TAMANHO, verbose=False)[0]
    p_sem = float(r.probs.data.cpu().numpy()[IDX_SEM])
    tem_lona = 0 if p_sem >= limiar else 1
    return {'tem_lona': tem_lona, 'classe': 'com_lona' if tem_lona else 'sem_lona',
            'confianca': round(max(p_sem, 1 - p_sem), 3)}

exemplo = caminhos[0]
print(os.path.basename(exemplo), '->', prever_lona(exemplo))

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

LIMIAR = 0.5
previsto = (prob_sem >= LIMIAR).astype(int)
print(classification_report(rotulos, previsto, target_names=['com_lona', 'sem_lona'], digits=3))
print('AUC:', round(roc_auc_score(rotulos, prob_sem), 3))
print('Matriz de confusão (linhas = real, colunas = previsto; ordem com_lona, sem_lona):')
print(confusion_matrix(rotulos, previsto))

print('\nFotos que o modelo errou (arquivo, classe real, prob. de sem_lona):')
for i in range(len(caminhos)):
    if previsto[i] != rotulos[i]:
        print('  ', os.path.basename(caminhos[i]), '|', CLASSES[rotulos[i]], '|', round(float(prob_sem[i]), 2))

In [ ]:
!pip -q install ultralytics

import os
from ultralytics import YOLO

caminho_drive = '/content/drive/MyDrive/classificador_lona.pt'
caminho_local = '/content/runs/final/weights/best.pt'

# Tenta usar o arquivo do Drive; se não achar, usa o arquivo local do treino recente
if os.path.exists(caminho_drive):
    modelo_path = caminho_drive
elif os.path.exists(caminho_local):
    modelo_path = caminho_local
else:
    raise FileNotFoundError("Não foi possível encontrar o arquivo do modelo treinado no Drive nem localmente.")

modelo = YOLO(modelo_path)
FOTOS = '/content/drive/MyDrive/Imagens Barreiras/Fotos_Testes'

certas, total = 0, 0
for pasta, _, arquivos in os.walk(FOTOS):
    for nome in sorted(arquivos):
        if not nome.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
            continue
        caminho = os.path.join(pasta, nome)
        r = modelo.predict(caminho, imgsz=224, verbose=False)[0]
        classe = r.names[int(r.probs.top1)]
        conf = float(r.probs.top1conf)
        # a resposta certa vem do começo do nome: com_lona... ou sem_lona...
        esperado = 'com lona' if nome.lower().startswith('com') else 'sem lona'
        previsto = 'sem lona' if 'sem' in classe.lower() else 'com lona'
        ok = (esperado == previsto)
        certas += ok
        total += 1
        print(f'{"OK  " if ok else "ERRO"} {os.path.relpath(caminho, FOTOS):40s} -> {previsto} ({conf:.0%})')

print(f'\nAcertos: {certas} de {total} ({certas/total:.0%})')